> **구분: 수업 핵심** — 표시된 시간 안에 강사와 함께 진행합니다.

# 03. Resize와 letterbox — 65분

**예상 시간: 65분**

이 노트북은 영상 처리를 처음 배우는 비전공 학부생을 기준으로 합니다. 설명을 외우기보다 **입력 → 배열 → 변환 → 저장 → 재검증** 순서를 따라가세요.

목표는 shape와 OpenCV size 순서를 구분하고, 종횡비를 유지한 canvas와 좌표 복원용 metadata를 만드는 것입니다.

필요한 실행·저장 규칙은 이 노트북에서 다시 제공합니다. [06.OpenCV README](README.md)와 [용어 찾아보기](GLOSSARY.md)는 추가 참고 자료입니다. TODO는 JupyterLab 파일 편집기에서 저장하고, 미완성 검증이 실패하는 것은 정상입니다. 참고 정답 셀은 강사가 안내하는 시점에 펼칩니다.

수업 경로: resize 15분 → letterbox 계산 15분 → TODO 25분 → 통합 확인 10분.


<!-- lesson-purpose:start -->
## 이 실습은 무엇을 위한 것인가요?

모델 입력 크기에 맞추려고 가로·세로를 따로 늘리면 물체 모양과 좌표가 달라집니다. letterbox는 비율을 유지해 줄이거나 늘리고 나머지를 padding으로 채웁니다. 예측 좌표를 원본 영상으로 돌리려면 이 변환의 기록이 필요합니다.

### 전체 과정에서 내 코드가 하는 일

```text
원본 H,W
  ↓
공통 scale
  ↓
반올림한 크기
  ↓
좌우·상하 padding
  ↓
canvas·metadata
  ↓
원본 좌표 복원
```

resize와 padding TODO를 구현하고 실제 영상 영역·색·shape까지 확인합니다. metadata 숫자만 맞추고 내용이 없는 canvas를 반환하면 완료가 아닙니다.

### 실행 전에 생각하고 결과로 설명하기

**질문:** canvas 좌표에서 padding만 빼면 원본 좌표인가요?

**설명:** 크기 비율도 되돌려야 합니다. padding을 뺀 뒤 scale로 나누며 정수 반올림의 영향도 고려합니다. 원본·강제 resize·letterbox를 비교해 모양을 보존한 쪽을 설명합니다.

이후의 TODO는 각 절에서 지정한 위치만 수정하고, 정답 해설의 이유를 자기 입력에 적용해 설명합니다. 예제 실행 성공·학생 함수 검증·실제 장치 관찰은 해당 절에서 요구하는 범위대로 나누어 기록합니다.
<!-- lesson-purpose:end -->


<!-- notebook-restart:start -->
## 실습 시작 확인

현재 노트북의 Python 커널과 `/workspace` 자료 경로를 확인합니다. 새 커널이거나 커널을 재시작했다면 첫 설정·import 셀부터 순서대로 실행합니다. 이전 노트북의 변수·import는 자동으로 전달되지 않습니다. 저장한 노트북·소스와 `Data` 파일은 유지됩니다. 접속·재부팅 복구는 [Setup 안내](../01.Setup/README.md)를 참고합니다.

**커널 준비:** JupyterLab에서 **Python 3 (ipykernel)** 을 선택합니다. 새 커널에서는 이 노트북의 첫 Python 셀부터 실행합니다. 다른 단원의 모듈이 남아 있는 커널이라면 재시작한 뒤 준비 셀을 다시 실행합니다. 실행 경로는 `/workspace` 또는 현재 단원 폴더이고 Python은 `/opt/course/bin/python`입니다. 다른 단원과 실행 중인 커널을 공유하지 않습니다. 각 단원의 `common` 모듈 이름이 같아 이전 import가 남으면 다른 코드를 읽을 수 있습니다. `NameError`가 나면 필요한 초기화 셀부터 다시 실행합니다.

TODO는 지정된 `exercises/*.py`를 JupyterLab 편집기에서 수정·저장하고 **학습자 검증 셀** 을 다시 실행합니다. 이 노트북 안의 TODO는 해당 코드 셀을 직접 수정합니다. 미완성 상태의 `NotImplementedError`·검증 실패는 예상된 출발점입니다. 참고 정답 검증 통과는 학습자 파일의 통과와 별도입니다.

**OpenCV 준비:** Ollama·ROS node는 필요 없습니다. 첫 Python 셀의 `prepare_assets()`가 합성 입력을 준비하고 검증합니다. 입력은 `Data/images/06.OpenCV/sample_scene.png`와 `Data/videos/06.OpenCV/sample_motion.avi`, 출력은 `Data/outputs/06.OpenCV`입니다. 01 노트북을 먼저 실행해 만든 결과가 없어도 시작할 수 있습니다. GUI·USB 카메라는 로컬 컨테이너에서 한 번만 실행하고 두 수강생이 로컬 화면을 함께 봅니다.

### 앞 실습의 위치와 이번에 다시 필요한 내용

[01_image_video_io.ipynb의 1절](01_image_video_io.ipynb)은 uint8 BGR 배열의 `shape=(높이,너비,3)`와 읽기·저장을, [02_colors_drawing.ipynb의 2절](02_colors_drawing.ipynb)은 표시 좌표 `(x,y)`와 원본 복사를 설명합니다. 이번에 필요한 내용은 **배열 shape는 H,W, OpenCV size는 W,H** 라는 차이와 원본을 바꾸지 않는다는 규칙입니다. 입력은 `load_image()`로 배포 합성 이미지를 다시 읽습니다. 앞에서 만든 gray·mask·overlay 결과는 필요 없습니다.

<!-- notebook-restart:end -->

## 슬라이드와 함께 진행하기

노트북의 설명·예제·TODO를 먼저 진행한 뒤, 아래 복귀 지점에서 결과를 슬라이드로 복습합니다. 쪽수는 표지를 포함한 실제 페이지 순서입니다.

- **수업 핵심 TODO부터 「최종 점검」까지** → [3일차 슬라이드 24쪽](https://docs.google.com/presentation/d/1dmIJ53VWrgEbYRjZqCYmUiRc9kOc_i7DZMNx6Y4a-sk/edit#slide=id.d3_s020) · [배포 PDF 24쪽](../슬라이드/3일차.pdf#page=24)에서 30쪽까지 복습합니다.

수업 핵심 범위와 지정된 완료 검사를 진행하고 선택·자습 확장은 수업 후에 이어갑니다. 복습에는 방금 만든 결과를 사용하며, 기존 수업 시간 안에서 진행합니다.


## 정답 활용 안내

> 교재 버전: v1.0

정답 확인 시점과 직접 시도 시간은 강사가 현장 진도·이해도에 맞춰 조절합니다. 검증 통과 전에도 강사 안내에 따라 정답을 참고할 수 있습니다. 정답과 해설을 비교한 뒤 자신의 TODO에 반영하고 다시 검증하세요. 정답 실행만으로 자신의 TODO가 완성되지는 않습니다.


> **구분: 수업 핵심**

수업 시작에 입력과 기대 결과를 예측하고, 종료 전 아래 두 완료 조건을 확인합니다.

## 이 노트북의 완료 확인과 단원 연결 질문

[작업대 상태 확인 사례](../공통사례.md) · [평가 워크북](../평가워크북.md)

- **이 노트북 완료:** 이 노트북에 표시된 핵심 TODO·검증·저장 결과를 확인합니다. 자습 노트북은 자습 범위만 확인합니다.
- **단원 전체 결과:** 원본을 보존하고 변환·영상 저장 결과를 다시 엽니다.
- **단원 마지막 연결 질문:** 같은 작업대 대상의 밝기 또는 크기만 바꿔 mask가 달라지는지 비교합니다. 색 임계값으로 충분한 조건과 어려운 조건을 한 가지씩 적습니다.

각 단원의 마무리·비교 시간에 5분을 사용합니다. 별도 시간을 추가하지 않습니다. 마지막 5분은 저장·오류 복구에 우선 사용하고 심화 설명은 자습으로 넘깁니다. 결과와 짧은 이유를 `Data/outputs/course-review/<단원>/`의 개인 기록에 남깁니다. 자료가 부족하면 미측정으로 적으며 성공값을 만들어 넣지 않습니다.

USB 카메라 실습은 로컬 환경에서 한 번만 실행하고 두 수강생이 로컬 화면을 함께 봅니다. 역할을 교대하고 각자 결과를 기록합니다. [공동 실습 안내](../공동카메라실습.md)를 따릅니다. 녹화 경로는 재생·자습·장애 대응용 선택 경로입니다.


In [ ]:
# 구분: 수업 핵심 셀
from pathlib import Path
import importlib, json, subprocess, sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / '06.OpenCV/common').is_dir())
COURSE = ROOT / '06.OpenCV'
if str(COURSE) not in sys.path: sys.path.insert(0, str(COURSE))
import cv2, numpy as np
from common.vision import SAMPLE_IMAGE, SAMPLE_VIDEO, OUTPUT_DIR, load_image, prepare_assets, probe_video, save_image
from common.contracts import check
asset_report = prepare_assets()
print('Python:', sys.executable)
print('OpenCV:', cv2.__version__, 'NumPy:', np.__version__)
print(json.dumps(asset_report, ensure_ascii=False, indent=2))


> **구분: 수업 핵심** — 표시된 시간 안에 강사와 함께 진행합니다.

## 1. 두 크기 순서에 유념하세요

- `image.shape[:2]`는 `(source_height, source_width)`입니다.
- `cv2.resize(image, dsize)`의 `dsize`는 `(target_width, target_height)`입니다.
- `VideoWriter`의 size도 `(width,height)`입니다.

`640×360` 이미지를 `(640,640)`로 강제 resize하면 출력 크기는 맞지만 세로만 늘어나 도형 비율이 바뀝니다. 모델이 학습한 모양과 달라질 수도 있습니다.


In [ ]:
# 구분: 수업 핵심 셀
image = load_image()
forced = cv2.resize(image, (320, 320), interpolation=cv2.INTER_AREA)
print('input HWC:', image.shape)
print('forced HWC:', forced.shape)
print('입력 width/height=', image.shape[1]/image.shape[0])
print('강제 resize width/height=', forced.shape[1]/forced.shape[0])


> **구분: 수업 핵심** — 표시된 시간 안에 강사와 함께 진행합니다.

## 2. Letterbox를 손으로 계산합니다

<!-- procedure:lesson-23-1236a5:start -->
![모델 좌표를 원본으로 되돌리기](../교재그림/절차/lesson-23.svg)

배율·padding·반올림 계약을 함께 저장합니다. 모델 영상 좌표를 원본 좌표처럼 쓰지 않습니다.
<!-- procedure:lesson-23-1236a5:end -->


입력 `(source_w,source_h)=(640,360)`, 목표 `(target_w,target_h)=(320,320)`이라면:

1. `scale = min(320/640, 320/360) = 0.5`
2. 새 크기 `(round(640×0.5), round(360×0.5)) = (320,180)`
3. 남는 가로는 0, 세로는 `320-180=140`
4. 위 70, 아래 70 pixel을 pad

홀수만큼 남으면 한쪽이 1 pixel 더 큽니다. `top + new_h + bottom == target_h`를 마지막에 검사하면 반올림 오류를 잡을 수 있습니다.


> **구분: 수업 핵심** — 표시된 시간 안에 강사와 함께 진행합니다.

### 실행 전 예측

세로가 긴 100×200 이미지(너비×높이)를 321×199 canvas에 넣으면 padding은 좌우와 상하 중 어디에 생길까요? 원본이 작은 경우 scale이 1보다 커질 수 있을까요?


> **구분: 수업 핵심** — 표시된 시간 안에 강사와 함께 진행합니다.

## 수업 핵심 TODO — [letterbox.py](exercises/letterbox.py)

수정 파일: [exercises/letterbox.py](exercises/letterbox.py)

가로·세로·정사각형과 홀수 padding에서 target shape를 정확히 만들고 scale·새 크기·네 padding을 반환하세요. 원본 배열은 변경하지 않습니다.

JupyterLab 파일 편집기에서 TODO를 구현하고 **저장** 한 뒤 아래 검증을 실행합니다. 검증은 저장된 모듈을 새 이름으로 다시 읽으므로 일반적으로 kernel 재시작은 필요 없습니다.

<!-- todo-review:requirements:start -->
### 작성 전에 확인할 입력·출력과 순서

**입력 → 출력:** `letterbox(image, target_size=(640,640), pad_color=(114,114,114))`는 BGR uint8 배열을 받아 `(canvas, info)`를 반환합니다. target_size는 **(width,height)** , canvas.shape는 **(height,width,3)** 입니다. info key는 `scale,new_width,new_height,left,top,right,bottom`입니다.
**작성 흐름 힌트:** 양수 크기 검사 → 두 축 확대 비율 중 작은 값 선택 → 새 폭·높이를 반올림하고 1 이상/목표 이하로 제한 → 축소는 INTER_AREA, 확대는 INTER_LINEAR → 남은 폭·높이 분배 → copyMakeBorder → 정보 dict 반환.
**손으로 확인:** 원본 W=4,H=2, 목표 W=5,H=5이면 scale=1.25, 새 크기 5×2(Python round), 좌우 0/0, 위아래 1/2입니다. 홀수 잔여 pixel은 right/bottom에 둡니다.
**설명할 지점:** “padding 양쪽을 모두 나눗셈 내림으로 계산하면 홀수일 때 한 pixel이 사라집니다. 반대쪽은 전체 잔여량에서 빼세요.”
**입력 검사·통과 기준:** image가 uint8 NumPy 배열인지, `(H,W,3)`이고 H·W가 양수인지 검사하고 잘못된 입력에는 `ValueError`를 냅니다. target_size는 (폭,높이)이며 두 값은 양수, pad_color는 BGR 순서입니다. starter에 `cv2`를 import합니다. `info['scale']`은 반올림 전 실수 배율, 크기·padding은 정수입니다. `check('letterbox','exercise')`가 가로·세로·정사각형 및 홀수 여백의 방향, 실제 리사이즈 영상·지정 padding 색, uint8·크기 정보와 원본 보존을 확인합니다. 아래 저장 예제는 자신의 함수를 새로 읽으므로 수정·저장 후 다시 실행할 수 있습니다.
<!-- todo-review:requirements:end -->


In [ ]:
# 구분: 수업 핵심 셀
print((COURSE/'exercises/letterbox.py').read_text())


In [ ]:
# 구분: 수업 핵심 셀
check('letterbox', 'exercise')
print('learner_verification=PASS')


> **구분: 수업 후 복습·자습** — 핵심 완료 뒤 원리와 확장을 확인합니다.

### 참고 정답

강사가 안내하는 시점에 정답과 해설을 비교합니다. 정답은 별도 파일이며 exercise를 덮어쓰지 않습니다. 구현이 한 글자씩 같을 필요는 없고 입력·출력·오류 처리 계약이 같으면 됩니다.


In [ ]:
# 구분: 수업 후 복습·자습 셀
print((COURSE/'solutions/letterbox.py').read_text())


### 정답 해설 — 입력부터 결과까지

**비율을 먼저 결정:** 목표 폭/원본 폭과 목표 높이/원본 높이 중 작은 비율을 선택하면 영상 전체가 목표 안에 들어갑니다. 두 축에 같은 scale을 사용하므로 물체 비율을 유지합니다. shape는 높이·폭 순이고 resize 인자는 폭·높이 순입니다.

640×480→640×640이면 scale=1, 새 영상 640×480, 위·아래 여백 각각 80입니다. 출력 좌표 (100,180)의 원본 위치는 여백을 빼고 scale로 나누어 (100,100)입니다.

여백이 홀수면 `//2`를 앞쪽에, 남은 값을 뒤쪽에 둬 정확한 전체 크기를 만듭니다. 축소에는 AREA, 확대에는 LINEAR를 선택합니다. meta의 scale·left·top은 좌표를 원본으로 돌릴 때 필요합니다.

**확인 질문:** 여백을 빼지 않고 scale만 나누면 검출 위치가 어느 방향으로 틀어지나요?

**적용·재검증:** 정답을 읽고 지정된 `exercises/` 파일의 TODO에 반영해 저장합니다. 노트북의 학습자 검증 셀을 다시 실행하고 경계 입력도 확인합니다. 참고 정답의 검증 통과는 학생 파일의 통과를 대신하지 않습니다. 이후 기존 실행·저장 셀에서 자신의 결과 경로를 확인합니다.

<!-- todo-review:answer:start -->
<details>
<summary>실행 없이 읽는 참고 정답과 단계별 주석</summary>

[실제 구현 파일](solutions/letterbox.py)에서 가져온 코드입니다. 이 Markdown 블록은 실행되지 않습니다. `solutions/`가 연결 어댑터인 단원은 아래 실제 구현을 읽고, 학생 파일에서는 기존 함수 이름과 인자를 유지하세요. 아래에 포함된 보조 함수·상수는 알고리즘을 읽기 위한 문맥입니다.

```python
"""03_resize_letterbox 참고 정답."""

import cv2
import numpy as np


def letterbox(
    image: np.ndarray,
    target_size: tuple[int, int] = (640, 640),
    pad_color: tuple[int, int, int] = (114, 114, 114),
) -> tuple[np.ndarray, dict[str, int | float]]:
    if not isinstance(image, np.ndarray) or image.dtype != np.uint8:
        raise ValueError("image는 uint8 numpy.ndarray여야 합니다.")
    if image.ndim != 3 or image.shape[2] != 3 or min(image.shape[:2]) <= 0:
        raise ValueError("image shape은 비어 있지 않은 (height, width, 3)이어야 합니다.")
    target_width, target_height = map(int, target_size)
    if target_width <= 0 or target_height <= 0:
        raise ValueError("target_size의 width와 height는 양수여야 합니다.")
    source_height, source_width = image.shape[:2]
    # 한 비율을 두 축에 같이 적용해 종횡비를 보존합니다.
    scale = min(target_width / source_width, target_height / source_height)
    new_width = max(1, min(target_width, round(source_width * scale)))
    new_height = max(1, min(target_height, round(source_height * scale)))
    interpolation = cv2.INTER_AREA if scale < 1.0 else cv2.INTER_LINEAR
    resized = cv2.resize(image, (new_width, new_height), interpolation=interpolation)
    horizontal = target_width - new_width
    vertical = target_height - new_height
    # 홀수 여백의 나머지를 오른쪽/아래에 둬 목표 크기를 정확히 맞춥니다.
    left, right = horizontal // 2, horizontal - horizontal // 2
    top, bottom = vertical // 2, vertical - vertical // 2
    canvas = cv2.copyMakeBorder(
        resized, top, bottom, left, right, cv2.BORDER_CONSTANT, value=pad_color
    )
    info = {
        "scale": float(scale), "new_width": new_width, "new_height": new_height,
        "left": left, "top": top, "right": right, "bottom": bottom,
    }
    return canvas, info
```

</details>
<!-- todo-review:answer:end -->


In [ ]:
# 구분: 수업 핵심 셀
check('letterbox', 'solution')
print('solution_verification=PASS')


In [ ]:
# 구분: 수업 핵심 셀
from common.contracts import _load
letterbox = _load('letterbox', 'exercise').letterbox
image = load_image()
canvas, info = letterbox(image, (640,640))
save_image(OUTPUT_DIR/'03_letterbox_640.png', canvas)
print('canvas:', canvas.shape, 'metadata:', info)
assert canvas.shape == (640,640,3)
assert info['top'] + info['new_height'] + info['bottom'] == 640

# 저장·검증한 배열을 PNG로 표시합니다. 카메라나 모델을 새로 실행하지 않습니다.
import base64
import html
from IPython.display import HTML, display

def show_image_comparison(images):
    cards = []
    for pixels, title in images:
        ok, encoded = cv2.imencode('.png', pixels)
        assert ok, '비교 영상의 PNG 인코딩 실패'
        uri = 'data:image/png;base64,' + base64.b64encode(encoded.tobytes()).decode('ascii')
        cards.append(f'<figure style="margin:8px;flex:1;min-width:220px">'
                     f'<img src="{uri}" style="width:100%;height:280px;object-fit:contain" '
                     f'alt="{html.escape(title)}"><figcaption>{html.escape(title)}</figcaption></figure>')
    display(HTML('<div style="display:flex;flex-wrap:wrap">' + ''.join(cards) + '</div>'))

show_image_comparison([(image, '원본'), (cv2.resize(image, (640, 640)), '강제 resize'), (canvas, '학생 함수의 letterbox')])


> **구분: 수업 핵심** — 표시된 시간 안에 강사와 함께 진행합니다.

## 3. 검출 좌표를 원본으로 되돌리기

<!-- procedure:lesson-23-5f9fc0:start -->
![모델 좌표를 원본으로 되돌리기](../교재그림/절차/lesson-23.svg)

배율·padding·반올림 계약을 함께 저장합니다. 모델 영상 좌표를 원본 좌표처럼 쓰지 않습니다.
<!-- procedure:lesson-23-5f9fc0:end -->


원본 x/y가 letterbox에서 `x_box=x_original×scale+left`, `y_box=y_original×scale+top`으로 이동합니다. 반대로는 padding을 뺀 뒤 scale로 나눕니다. padding 영역에서 나온 box나 canvas 경계 밖 값은 원본 크기로 clip해야 합니다. 이 단원 TODO는 metadata를 반환해 이후 YOLO 결과 복원에 쓸 근거를 남깁니다.


> **구분: 수업 핵심** — 표시된 시간 안에 강사와 함께 진행합니다.

## 최종 점검

- 직접 resize는 내용 비율을 바꿀 수 있고 letterbox는 빈 영역을 추가합니다.
- interpolation 선택은 속도·화질에 영향을 주지만 원본에 없던 실제 정보를 만들지는 않습니다.
- 목표 shape 하나만 검사하지 않고 scale, padding 합, 원본 불변성, 가로·세로 입력을 함께 검사합니다.
- 이 노트북은 letterbox TODO 검증과 `03_letterbox_640.png` 저장으로 완료합니다. 단원 전체에서는 01의 `01_gray.png`·`video_pipeline.json`(36프레임), 02의 `02_red_mask.png`·`02_red_overlay.png`와 밝기 조건 기록도 확인합니다. 앞 노트북 산출물은 단원 완료 증거이며 현재 letterbox의 입력은 아닙니다.




<!-- slide-sync:start -->
## 슬라이드로 돌아가기 — 결과 복습

**완료 확인:** letterbox 출력·metadata·좌표 복원 확인

[3일차 슬라이드 24쪽](https://docs.google.com/presentation/d/1dmIJ53VWrgEbYRjZqCYmUiRc9kOc_i7DZMNx6Y4a-sk/edit#slide=id.d3_s020) · [배포 PDF 24쪽](../슬라이드/3일차.pdf#page=24)로 돌아가 **24–30쪽** 에서 방금 수행한 내용과 결과를 설명합니다.
<!-- slide-sync:end -->

> **구분: 수업 후 복습·자습** — 핵심 완료 뒤 원리와 확장을 확인합니다.

## 자습 — interpolation과 좌표 반올림

축소에는 영역 평균에 가까운 `INTER_AREA`, 일반 확대에는 `INTER_LINEAR`가 흔합니다. segmentation mask나 class ID map은 중간 범주를 만들지 않도록 nearest-neighbor가 필요할 수 있습니다. 좌표를 너무 일찍 정수로 바꾸면 반복 변환에서 오차가 커지므로 마지막 drawing 단계까지 float를 유지하는 방식을 비교해 보세요.
